# TFG V7.1: Hardware-Ready Trotterised Quantum Walk with Defect

This notebook is the hardware-oriented version of V7. It keeps the continuous-time quantum-walk idea, but removes the two shortcuts used by the exact spectral prototype:

- the circuit is not built from a dense exact `UnitaryGate(exp(-iHt))`;
- the circuit is not given `valid_indices` as a precomputed solution list.

Instead, valid windows are detected by a reversible oracle that loads the candidate window into a work register, applies a phase only when that window is all zero, and uncomputes the work register. The graph hopping is implemented digitally with local two-level mixer gates over neighbouring window-start indices.


## Mathematical and Algorithmic Setup

The search space is the set of candidate window starts. The index register stores one start position $i$, the grid register stores the input occupancy bitstring $g$, and the work register temporarily stores the contents of the selected window.

The target continuous-time Hamiltonian is approximated by first-order Trotter steps. For the adjacency convention,

$$H_{\mathrm{adj}}=-\gamma A-\lambda P_V,$$

so one digital step is

$$e^{-iH\Delta t}\approx e^{i\gamma A\Delta t}e^{i\lambda P_V\Delta t}.$$

For the Laplacian convention,

$$H_{\mathrm{lap}}=\gamma(D-A)-\lambda P_V,$$

so the step adds a diagonal degree phase $e^{-i\gamma D\Delta t}$. The important difference from V7 is that $P_V$ is not supplied as a list of valid indices; it is implemented by a reversible validity oracle.


## Requirements and Imports

The notebook uses only standard Qiskit circuit primitives plus NumPy and Matplotlib. Statevector simulation is used only for small demonstrations because the hardware-ready register layout includes the grid and work registers, so the Hilbert space grows with the input grid size.


In [1]:

# pip install qiskit qiskit-aer qiskit-ibm-runtime numpy matplotlib

import os
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))
os.environ.setdefault("MPLBACKEND", "Agg")

from math import prod
from pathlib import Path
import csv
import re
import time

import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import MCXGate, PhaseGate

try:
    from qiskit_aer import AerSimulator
except Exception:
    AerSimulator = None

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

VALID_COLOR = "#2ecc71"
INVALID_COLOR = "#e74c3c"
BASELINE_COLOR = "0.45"

V7_OUTPUT_DIR = Path("TFG_V7_1_Analysis")
V7_OUTPUT_DIR.mkdir(exist_ok=True)

# V7 uses the earliest time whose success probability is within this
# absolute tolerance of the best scanned peak. This keeps the walk fast
# while preserving a strong success-probability advantage.
V7_PEAK_TOLERANCE = 0.05
V7_1_MAX_STATEVECTOR_QUBITS = 16
V7_1_MAX_SIM_QUBITS = 100
V7_1_DEFAULT_TROTTER_STEPS = 4
V7_1_AER_METHOD = "matrix_product_state"
V7_1_MAX_AER_OPS = 80_000
V7_1_LARGE_CASE_SHOTS = 128
V7_1_LARGE_CASE_OP_THRESHOLD = 25_000

# If True, each case scans only times t <= r_Grover = (pi/4)*sqrt(W/K).
# This forces the quantum-walk search to use no more time/iteration scale
# than the standard Grover estimate for the same valid fraction.
V7_ENFORCE_GROVER_TIME_CAP = True

V7_DEFAULT_T_MAX = 200.0
V7_T_STEPS = 4000

print("qiskit version:", qiskit.__version__)


qiskit version: 2.3.1


## Utility Functions

These are the same geometry utilities used throughout the TFG versions. They define row-major coordinate conversion, candidate window starts, window cell lookup, and the uniform preparation over the physical candidate-index range $0,\ldots,W-1$.


In [2]:

# =========================================================
# ND geometry and classical utilities
# =========================================================

def validate_problem(N, M):
    if len(N) != len(M):
        raise ValueError("N and M must have the same dimension.")
    for d, (n_d, m_d) in enumerate(zip(N, M)):
        if n_d <= 0 or m_d <= 0:
            raise ValueError(f"N[{d}] and M[{d}] must be positive.")
        if m_d > n_d:
            raise ValueError(f"M[{d}] cannot be greater than N[{d}].")


def coord_to_index(coord, dims):
    """Converts ND coordinates to a row-major linear index.
    Example with dims = [4,4]:
    (0,0) -> 0    (0,1) -> 1    (0,2) -> 2    (0,3) -> 3
    (1,0) -> 4    (1,1) -> 5    (1,2) -> 6    (1,3) -> 7
    (2,0) -> 8    (2,1) -> 9    (2,2) -> 10   (2,3) -> 11
    (3,0) -> 12   (3,1) -> 13   (3,2) -> 14   (3,3) -> 15
    """
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def index_to_coord(index, dims):
    """Converts a row-major linear index to ND coordinates.
    Example with dims = [4,4]:
    0 -> (0,0)    1 -> (0,1)    2 -> (0,2)    3 -> (0,3)
    4 -> (1,0)    5 -> (1,1)    6 -> (1,2)    7 -> (1,3)
    8 -> (2,0)    9 -> (2,1)    10 -> (2,2)   11 -> (2,3)
    12 -> (3,0)   13 -> (3,1)    14 -> (3,2)   15 -> (3,3)
    """
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)


def valid_starts_nd(N, M):
    """Valid starting coordinates for a window M inside N."""
    return list(np.ndindex(tuple(N[d] - M[d] + 1 for d in range(len(N)))))


def window_qubits_nd(start, N, M):
    """Linear indices of the cells covered by the window starting at start."""
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def normalize_coord(coord, D):
    if D == 1 and isinstance(coord, int):
        return (coord,)
    return tuple(coord)


def build_grid_bits(N, occupied_coords):
    """Returns a classical vector with 1 on occupied cells and 0 on free cells."""
    D = len(N)
    grid = [0] * prod(N)
    for raw_coord in occupied_coords:
        coord = normalize_coord(raw_coord, D)
        if len(coord) != D:
            raise ValueError(f"Coordinate {coord} does not have dimension {D}.")
        for d, x in enumerate(coord):
            if x < 0 or x >= N[d]:
                raise ValueError(f"Coordinate {coord} is outside the grid N={N}.")
        grid[coord_to_index(coord, N)] = 1
    return grid


def compute_window_cost_classical(grid_bits, start, N, M):
    """C(i): number of ones in the window associated with start."""
    return sum(grid_bits[q] for q in window_qubits_nd(start, N, M))



def gray_order_valid(W, IDX):
    """
    000 -> 0
    001 -> 1
    011 -> 3
    010 -> 2
    110 -> 6
    111 -> 7
    101 -> 5
    100 -> 4
    """
    gray_full = [t ^ (t >> 1) for t in range(2**IDX)]
    return [g for g in gray_full if g < W]


def format_nd_array_from_bits(bitstring, dims):
    arr = np.array(list(bitstring), dtype=str).reshape(tuple(dims))
    return np.array2string(arr, separator=' ').replace("'", "")


# =========================================================
# Circuit index construction utility
# =========================================================

def prepare_valid_index_superposition(qc, idx, W):
    IDX = len(idx)
    amps = np.zeros(2**IDX, dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


## Case Studies

The case definitions are kept identical to V7.1's parent V7 notebook. The hardware circuit builder will use only `N`, `M`, and `occupied_coords`; valid indices are computed only in reporting helpers after simulation, never passed to the quantum oracle.


In [3]:

CASE_STUDIES = [
    {
        "name": "01_1d_tiny_single_gap",
        "description": "Minimal 1D case with one valid window.",
        "N": [6], "M": [2],
        "occupied_coords": [(0,), (3,), (4,)],
        "theta": np.pi / 2, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "02_1d_main_reference",
        "description": "Reference 1D instance used in the manual main experiment.",
        "N": [8], "M": [2],
        "occupied_coords": [(0,), (1,), (2,), (6,), (7,)],
        "theta": np.pi / 3, "beta": 0.60, "mixer_method": "local_geometric",
    },
    {
        "name": "03_1d_two_free_regions",
        "description": "1D grid with two occupied blocks and a central valid plateau.",
        "N": [10], "M": [3],
        "occupied_coords": [(0,), (1,), (7,), (8,), (9,)],
        "theta": np.pi / 3, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "04_1d_clustered_medium",
        "description": "Medium 1D case with several clustered obstacles.",
        "N": [16], "M": [3],
        "occupied_coords": [(0,), (1,), (5,), (6,), (7,), (13,), (14,)],
        "theta": np.pi / 3, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "05_1d_long_clustered_blocks",
        "description": "Longer 1D case with separated occupied blocks.",
        "N": [32], "M": [4],
        "occupied_coords": [(0,), (1,), (2,), (9,), (10,), (11,), (18,), (19,), (28,), (29,), (30,), (31,)],
        "theta": np.pi / 4, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "06_2d_tiny_corner_block",
        "description": "Small 2D case with a single valid 2x2 region.",
        "N": [3, 3], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 2), (2, 2)],
        "theta": np.pi / 2, "beta": 0.28, "mixer_method": "local_geometric",
    },
    {
        "name": "07_2d_small_diagonal_block",
        "description": "4x4 grid with diagonal obstacles and one clear lower-left solution.",
        "N": [4, 4], "M": [2, 2],
        "occupied_coords": [(1, 1), (2, 2), (0, 3)],
        "theta": np.pi / 2, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "08_2d_medium_clustered_obstacles",
        "description": "5x5 grid with two compact occupied clusters.",
        "N": [5, 5], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (3, 3), (3, 4), (4, 3)],
        "theta": np.pi / 3, "beta": 0.22, "mixer_method": "local_geometric",
    },
    {
        "name": "09_2d_rectangular_window",
        "description": "6x6 grid with a non-square 3x2 window.",
        "N": [6, 6], "M": [3, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (4, 4), (4, 5), (5, 4), (2, 3)],
        "theta": np.pi / 3, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "10_3d_small_clustered_obstacles",
        "description": "Small 3D grid with two compact occupied clusters.",
        "N": [4, 4, 3], "M": [2, 2, 2],
        "occupied_coords": [(0, 0, 0), (0, 0, 1), (1, 0, 0), (3, 3, 2), (3, 2, 2), (2, 3, 2)],
        "theta": np.pi / 4, "beta": 0.16, "mixer_method": "local_geometric",
    },
]


In [4]:
def qw_slug(text):
    """Return a filesystem-safe lowercase slug for figure filenames."""
    return re.sub(r"[^a-zA-Z0-9_]+", "_", str(text)).strip("_").lower()


def save_qw_figure(fig, stem):
    """Save a Matplotlib figure as PDF and PNG in the V7.1 analysis folder."""
    # Guardamos figuras separadas para no mezclar prototipo exacto y version hardware.
    V7_OUTPUT_DIR.mkdir(exist_ok=True)
    fig.savefig(V7_OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(V7_OUTPUT_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def qw_case_context_hardware(case):
    """Build a case context without storing precomputed valid_indices."""
    # El contexto contiene solo geometria, grid de entrada y tamanos de registros.
    N_case = list(case["N"])
    M_case = list(case["M"])
    occupied_case = list(case["occupied_coords"])
    validate_problem(N_case, M_case)
    starts_case = valid_starts_nd(N_case, M_case)
    grid_bits_case = build_grid_bits(N_case, occupied_case)
    W_case = len(starts_case)
    IDX_case = int(np.ceil(np.log2(W_case))) if W_case > 1 else 1
    if W_case > 2**IDX_case:
        raise ValueError(f"W={W_case} does not fit in IDX={IDX_case} qubits.")
    return {
        "name": case["name"],
        "description": case.get("description", ""),
        "N": N_case,
        "M": M_case,
        "occupied_coords": occupied_case,
        "starts": starts_case,
        "grid_bits": grid_bits_case,
        "W": W_case,
        "IDX": IDX_case,
        "GRID": prod(N_case),
        "WIN": prod(M_case),
    }


def reporting_solution_indices(ctx):
    """Compute valid indices only for classical reporting and simulator scoring."""
    # Esta funcion no forma parte del circuito: solo sirve para comprobar la salida.
    return [
        i for i, start in enumerate(ctx["starts"])
        if compute_window_cost_classical(ctx["grid_bits"], start, ctx["N"], ctx["M"]) == 0
    ]


CASE_CONTEXTS = [qw_case_context_hardware(case) for case in CASE_STUDIES]
print(f"Loaded {len(CASE_CONTEXTS)} hardware contexts.")
for ctx in CASE_CONTEXTS:
    solution_for_report = reporting_solution_indices(ctx)
    print(
        f"{ctx['name']} | W={ctx['W']} | grid qubits={ctx['GRID']} | "
        f"work qubits={ctx['WIN']} | K(reporting only)={len(solution_for_report)}"
    )


Loaded 10 hardware contexts.
01_1d_tiny_single_gap | W=5 | grid qubits=6 | work qubits=2 | K(reporting only)=1
02_1d_main_reference | W=7 | grid qubits=8 | work qubits=2 | K(reporting only)=2
03_1d_two_free_regions | W=8 | grid qubits=10 | work qubits=3 | K(reporting only)=3
04_1d_clustered_medium | W=14 | grid qubits=16 | work qubits=3 | K(reporting only)=4
05_1d_long_clustered_blocks | W=29 | grid qubits=32 | work qubits=4 | K(reporting only)=11
06_2d_tiny_corner_block | W=4 | grid qubits=9 | work qubits=4 | K(reporting only)=1
07_2d_small_diagonal_block | W=9 | grid qubits=16 | work qubits=4 | K(reporting only)=1
08_2d_medium_clustered_obstacles | W=16 | grid qubits=25 | work qubits=4 | K(reporting only)=9
09_2d_rectangular_window | W=20 | grid qubits=36 | work qubits=6 | K(reporting only)=8
10_3d_small_clustered_obstacles | W=18 | grid qubits=48 | work qubits=8 | K(reporting only)=12


## Graph and Trotter Utilities

The graph topology is allowed to be known classically because it depends only on the problem dimensions, not on the solution. The adjacency convention uses the edge mixer directly. The Laplacian convention uses the same edge mixer plus an extra phase depending on each node degree.


In [5]:
def build_adjacency_matrix(starts, N):
    """Build the symmetric adjacency matrix of the window-start graph."""
    # Mapeamos coordenadas a indices para buscar vecinos en tiempo constante.
    starts = [tuple(start) for start in starts]
    W = len(starts)
    D = len(N)
    start_to_index = {start: i for i, start in enumerate(starts)}
    A = np.zeros((W, W), dtype=np.float64)
    for i, start in enumerate(starts):
        for d in range(D):
            for step in (-1, 1):
                neighbour = list(start)
                neighbour[d] += step
                neighbour = tuple(neighbour)
                j = start_to_index.get(neighbour)
                if j is not None:
                    A[i, j] = 1.0
                    A[j, i] = 1.0
    return A


def build_graph_laplacian(A):
    """Return L=D-A and the normalised Laplacian D^{-1/2}LD^{-1/2}."""
    # Calculamos grados y evitamos divisiones por cero en nodos aislados.
    A = np.asarray(A, dtype=np.float64)
    degrees = A @ np.ones(A.shape[0], dtype=np.float64)
    L = np.diag(degrees) - A
    inv_sqrt_degrees = np.zeros_like(degrees)
    nonzero = degrees > 0
    inv_sqrt_degrees[nonzero] = 1.0 / np.sqrt(degrees[nonzero])
    D_inv_sqrt = np.diag(inv_sqrt_degrees)
    return L, D_inv_sqrt @ L @ D_inv_sqrt


def adjacency_edges(A):
    """Return each undirected graph edge once."""
    # Extraemos la parte triangular superior para no duplicar rotaciones.
    rows, cols = np.where(np.triu(A, k=1) > 0.5)
    return list(zip(rows.tolist(), cols.tolist()))


def two_level_rotation_matrix(num_qubits, a, b, angle):
    """Return exp(-i angle X_ab) embedded in the index-register space."""
    # Esta es una primitiva compacta para prototipos pequenos; en hardware real se transpila.
    dim = 2**num_qubits
    U = np.eye(dim, dtype=complex)
    c = np.cos(float(angle))
    s = -1j * np.sin(float(angle))
    U[a, a] = c
    U[b, b] = c
    U[a, b] = s
    U[b, a] = s
    return U


def append_two_level_rotation(qc, idx, a, b, angle, label):
    """Append one local two-level mixer rotation on the index register."""
    # La matriz actua solo sobre idx; no toca grid ni work.
    from qiskit.circuit.library import UnitaryGate
    U = two_level_rotation_matrix(len(idx), a, b, angle)
    qc.append(UnitaryGate(U, label=label), list(idx))


## Reversible Validity Oracle

This is the main hardware-ready change. The oracle does not know which indices are valid. It computes the window contents for the current superposed index, applies a phase to the all-zero work state, and uncomputes the work register.

The obstacle grid is still the problem input. In this notebook it is prepared as a computational basis state using `X` gates on occupied cells; on real hardware this would be the state-preparation step for the given instance.


In [6]:
def append_mcx(qc, controls, target):
    """Append a multi-controlled X gate using Qiskit's MCXGate primitive."""
    # Tratamos los casos pequenos directamente para que los circuitos sean legibles.
    controls = list(controls)
    if len(controls) == 0:
        qc.x(target)
    elif len(controls) == 1:
        qc.cx(controls[0], target)
    else:
        qc.append(MCXGate(num_ctrl_qubits=len(controls)), controls + [target])


def apply_window_loader(qc, grid, idx, work, starts, N, M, order_valid):
    """Load the selected window into work by reversible XOR.

    Implements |g>|i>|m> -> |g>|i>|m xor window_i>. Applying the same block
    twice uncomputes the work register. The loop uses all candidate starts, not
    the valid starts.
    """
    # Recorremos los indices en orden Gray para reducir cambios X en los controles idx.
    IDX = len(idx)
    current_zero_mask = [False] * IDX
    for i in order_valid:
        bits = [(i >> b) & 1 for b in range(IDX)]
        target_zero_mask = [bits[b] == 0 for b in range(IDX)]
        for b in range(IDX):
            if current_zero_mask[b] != target_zero_mask[b]:
                qc.x(idx[b])
                current_zero_mask[b] = target_zero_mask[b]
        for j, grid_pos in enumerate(window_qubits_nd(starts[i], N, M)):
            controls = [idx[b] for b in range(IDX)] + [grid[grid_pos]]
            append_mcx(qc, controls, work[j])
    for b in range(IDX):
        if current_zero_mask[b]:
            qc.x(idx[b])


def apply_phase_to_basis_state(qc, register, basis_state, angle, label=None):
    """Apply exp(i angle) to one computational basis state of a register."""
    # Convertimos temporalmente el estado objetivo en |11...1>.
    if abs(float(angle)) < 1e-15:
        return
    qubits = list(register)
    zero_qubits = [q for bit, q in enumerate(qubits) if ((int(basis_state) >> bit) & 1) == 0]
    for q in zero_qubits:
        qc.x(q)
    if len(qubits) == 1:
        qc.p(float(angle), qubits[0])
    else:
        gate = PhaseGate(float(angle), label=label).control(len(qubits) - 1)
        qc.append(gate, qubits[:-1] + [qubits[-1]])
    for q in reversed(zero_qubits):
        qc.x(q)


def apply_all_zero_window_phase(qc, work, angle, label=None):
    """Apply exp(i angle) iff all work qubits are zero."""
    # El defecto se activa exactamente cuando la ventana cargada es 0...0.
    apply_phase_to_basis_state(qc, work, 0, angle, label=label)


def apply_defect_oracle_layer(qc, grid, idx, work, ctx, lam, dt, layer):
    """Apply the reversible phase oracle e^{i lambda P_V dt}."""
    # Cargamos ventana, marcamos work=0...0 y deshacemos la carga.
    order_valid = gray_order_valid(ctx["W"], ctx["IDX"])
    apply_window_loader(qc, grid, idx, work, ctx["starts"], ctx["N"], ctx["M"], order_valid)
    apply_all_zero_window_phase(qc, work, float(lam) * float(dt), label=f"valid-phase[{layer}]")
    apply_window_loader(qc, grid, idx, work, ctx["starts"], ctx["N"], ctx["M"], order_valid)


## Hardware-Ready Quantum Walk Circuit

The circuit uses three registers:

- `g`: the input grid bits;
- `i`: the candidate window-start index;
- `m`: a temporary work register containing the selected window.

No dense exact Hamiltonian is constructed. No precomputed `valid_indices` list is passed into the circuit. The only classical information compiled into the oracle is the grid geometry and the input bitstring.


In [7]:
def apply_degree_phase_layer(qc, idx, degrees, gamma, dt, layer):
    """Apply the Laplacian degree phase exp(-i gamma D dt)."""
    # Cada estado |i> acumula una fase proporcional al grado del nodo i.
    for i, degree in enumerate(degrees):
        apply_phase_to_basis_state(qc, idx, i, -float(gamma) * float(degree) * float(dt), label=f"deg[{layer},{i}]")


def apply_edge_mixer_layer(qc, idx, edges, gamma, dt, layer):
    """Apply first-order edge rotations for exp(+i gamma A dt)."""
    # Como la primitiva es exp(-i angle X_ab), usamos angle=-gamma*dt.
    for edge_id, (a, b) in enumerate(edges):
        append_two_level_rotation(qc, idx, a, b, -float(gamma) * float(dt), label=f"mix[{layer},{edge_id}]")


def build_hardware_qw_circuit(ctx, lam, t_total, r_trotter=4, gamma=1.0, mode="adjacency"):
    """Build a Trotterised CTQW circuit without valid_indices or exact exp(-iHt)."""
    # El circuito evalua la validez mediante oraculo reversible en cada capa.
    mode = str(mode).lower()
    if mode not in {"adjacency", "laplacian"}:
        raise ValueError("mode must be 'adjacency' or 'laplacian'.")
    r_trotter = int(r_trotter)
    if r_trotter <= 0:
        raise ValueError("r_trotter must be positive.")
    dt = float(t_total) / r_trotter

    grid = QuantumRegister(ctx["GRID"], "g")
    idx = QuantumRegister(ctx["IDX"], "i")
    work = QuantumRegister(ctx["WIN"], "m")
    qc = QuantumCircuit(grid, idx, work)

    for q, bit in zip(grid, ctx["grid_bits"]):
        if bit:
            qc.x(q)
    prepare_valid_index_superposition(qc, idx, ctx["W"])

    A = build_adjacency_matrix(ctx["starts"], ctx["N"])
    edges = adjacency_edges(A)
    degrees = A @ np.ones(ctx["W"], dtype=np.float64)

    for layer in range(r_trotter):
        if mode == "laplacian":
            apply_degree_phase_layer(qc, idx, degrees, gamma, dt, layer)
        apply_edge_mixer_layer(qc, idx, edges, gamma, dt, layer)
        apply_defect_oracle_layer(qc, grid, idx, work, ctx, lam, dt, layer)
        qc.barrier(label=f"Trotter {layer + 1}/{r_trotter}")

    metadata = {
        "mode": mode,
        "lam": float(lam),
        "gamma": float(gamma),
        "t_total": float(t_total),
        "r_trotter": r_trotter,
        "dt": dt,
        "edges": edges,
        "degrees": degrees,
        "qubits": qc.num_qubits,
        "operations": dict(qc.count_ops()),
    }
    return qc, metadata


## Small-Case Simulation and Scoring

The scoring code below computes valid indices only after building the circuit, and only to measure success probability in the simulator. This keeps the algorithmic path honest: the quantum circuit itself never receives the solution set.


In [8]:
def marginal_index_probabilities(sv, qc, idx_register, W):
    """Return marginal probabilities over the physical index states 0..W-1."""
    # Qiskit marginaliza sobre los qubits idx aunque existan grid y work.
    qargs = [qc.qubits.index(q) for q in idx_register]
    probs_full = np.asarray(sv.probabilities(qargs=qargs), dtype=float)
    return probs_full[:W]


def simulate_hardware_qw_case(ctx, lam=None, t_total=None, r_trotter=V7_1_DEFAULT_TROTTER_STEPS, gamma=1.0, mode="adjacency"):
    """Build and simulate one small hardware-layout QW case."""
    # La simulacion se limita a casos pequenos porque incluye grid+idx+work.
    solution_for_report = reporting_solution_indices(ctx)
    if not solution_for_report:
        raise ValueError("No valid windows in this case.")
    if lam is None:
        lam = np.sqrt(ctx["W"]) * float(gamma)
    if t_total is None:
        t_total = np.pi / 4.0 * np.sqrt(ctx["W"] / len(solution_for_report))

    qc, metadata = build_hardware_qw_circuit(
        ctx, lam=lam, t_total=t_total, r_trotter=r_trotter, gamma=gamma, mode=mode
    )
    if qc.num_qubits > V7_1_MAX_STATEVECTOR_QUBITS:
        raise ValueError(f"Circuit has {qc.num_qubits} qubits; statevector cap is {V7_1_MAX_STATEVECTOR_QUBITS}.")

    sv = Statevector.from_instruction(qc)
    idx_reg = qc.qregs[1]
    probs = marginal_index_probabilities(sv, qc, idx_reg, ctx["W"])
    p_valid = float(np.sum(probs[solution_for_report]))
    return {
        "case": ctx["name"],
        "ctx": ctx,
        "qc": qc,
        "metadata": metadata,
        "solution_indices_reporting_only": solution_for_report,
        "probs": probs,
        "P_valid": p_valid,
        "P_uniform": len(solution_for_report) / ctx["W"],
    }


## Demonstration Runs

The demonstration runs only the cases that fit the configured statevector cap. Larger cases still produce resource estimates and are ready for transpilation or hardware execution, but they are not statevector-simulated here.


In [9]:
SIM_RESULTS = []
for ctx in CASE_CONTEXTS:
    total_qubits = ctx["GRID"] + ctx["IDX"] + ctx["WIN"]
    if total_qubits > V7_1_MAX_STATEVECTOR_QUBITS:
        print(f"skip statevector reference: {ctx['name']} has {total_qubits} qubits")
        continue
    for mode in ["adjacency", "laplacian"]:
        result = simulate_hardware_qw_case(ctx, mode=mode)
        SIM_RESULTS.append(result)
        print(
            f"{ctx['name']} | mode={mode} | qubits={result['qc'].num_qubits} | "
            f"P_uniform={result['P_uniform']:.4f} | P_valid(statevector)={result['P_valid']:.4f} | "
            f"ops={result['metadata']['operations']}"
        )


01_1d_tiny_single_gap | mode=adjacency | qubits=11 | P_uniform=0.2000 | P_valid(statevector)=0.5325 | ops={'x': 99, 'mcx': 80, 'unitary': 16, 'cp': 4, 'barrier': 4, 'initialize': 1}


01_1d_tiny_single_gap | mode=laplacian | qubits=11 | P_uniform=0.2000 | P_valid(statevector)=0.5597 | ops={'x': 179, 'mcx': 80, 'mcphase': 20, 'unitary': 16, 'cp': 4, 'barrier': 4, 'initialize': 1}


02_1d_main_reference | mode=adjacency | qubits=13 | P_uniform=0.2857 | P_valid(statevector)=0.5668 | ops={'x': 117, 'mcx': 112, 'unitary': 24, 'cp': 4, 'barrier': 4, 'initialize': 1}


02_1d_main_reference | mode=laplacian | qubits=13 | P_uniform=0.2857 | P_valid(statevector)=0.4923 | ops={'x': 213, 'mcx': 112, 'mcphase': 28, 'unitary': 24, 'cp': 4, 'barrier': 4, 'initialize': 1}


03_1d_two_free_regions | mode=adjacency | qubits=16 | P_uniform=0.3750 | P_valid(statevector)=0.6738 | ops={'mcx': 192, 'x': 125, 'unitary': 28, 'mcphase': 4, 'barrier': 4, 'initialize': 1}


03_1d_two_free_regions | mode=laplacian | qubits=16 | P_uniform=0.3750 | P_valid(statevector)=0.6065 | ops={'x': 221, 'mcx': 192, 'mcphase': 36, 'unitary': 28, 'barrier': 4, 'initialize': 1}
skip statevector reference: 04_1d_clustered_medium has 23 qubits
skip statevector reference: 05_1d_long_clustered_blocks has 41 qubits


06_2d_tiny_corner_block | mode=adjacency | qubits=15 | P_uniform=0.2500 | P_valid(statevector)=0.7798 | ops={'mcx': 128, 'x': 83, 'unitary': 16, 'mcphase': 4, 'barrier': 4, 'initialize': 1}


06_2d_tiny_corner_block | mode=laplacian | qubits=15 | P_uniform=0.2500 | P_valid(statevector)=0.7798 | ops={'mcx': 128, 'x': 115, 'cp': 16, 'unitary': 16, 'mcphase': 4, 'barrier': 4, 'initialize': 1}
skip statevector reference: 07_2d_small_diagonal_block has 24 qubits
skip statevector reference: 08_2d_medium_clustered_obstacles has 33 qubits
skip statevector reference: 09_2d_rectangular_window has 47 qubits
skip statevector reference: 10_3d_small_clustered_obstacles has 61 qubits


## Hardware-Ready Comparison Graph

This graph compares the simulated success probability for all small cases that fit in the statevector cap. The probability baseline is still classical reporting, but the circuit execution path uses only the reversible oracle.


In [10]:
if SIM_RESULTS:
    cases = sorted({res["case"] for res in SIM_RESULTS})
    x = np.arange(len(cases))
    width = 0.25
    adj = {res["case"]: res for res in SIM_RESULTS if res["metadata"]["mode"] == "adjacency"}
    lap = {res["case"]: res for res in SIM_RESULTS if res["metadata"]["mode"] == "laplacian"}
    uniform_vals = np.array([adj.get(c, lap.get(c))["P_uniform"] for c in cases])
    adj_vals = np.array([adj[c]["P_valid"] if c in adj else np.nan for c in cases])
    lap_vals = np.array([lap[c]["P_valid"] if c in lap else np.nan for c in cases])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width, uniform_vals, width=width, color=BASELINE_COLOR, alpha=0.75, label="uniform")
    ax.bar(x, adj_vals, width=width, color=VALID_COLOR, alpha=0.9, label="adjacency Trotter")
    ax.bar(x + width, lap_vals, width=width, color="#3498db", alpha=0.85, label="laplacian Trotter")
    ax.set_xticks(x)
    ax.set_xticklabels([c.split("_", 1)[0] for c in cases])
    ax.set_xlabel("case")
    ax.set_ylabel("P_valid")
    ax.set_title("V7.1 hardware-ready Trotter QW: small-case simulation")
    ax.legend(fontsize=10)
    fig.tight_layout()
    save_qw_figure(fig, "v7_1_hardware_trotter_small_case_comparison")
else:
    print("No cases fit the current statevector simulation cap.")


## Shot-Based Measurement of the Index Register

The previous section uses an ideal statevector to read exact probabilities. This section follows the experimental workflow more closely: it measures the `idx` register, obtains finite-shot counts, and estimates the valid-window success probability as an observed frequency.

The solution indices are still computed classically only after measurement, for scoring the bitstrings. They are not used to build the circuit or the validity oracle.


In [11]:
V7_1_SHOTS = 4096
V7_1_SHOT_SEED = 12345


def add_index_measurements(qc):
    """Return a copy of qc with measurements on the index register only."""
    # Medimos solo idx porque el objetivo del algoritmo es devolver un indice de ventana.
    measured = qc.copy()
    idx_reg = measured.qregs[1]
    c_idx = ClassicalRegister(len(idx_reg), "c_i")
    measured.add_register(c_idx)
    measured.measure(idx_reg, c_idx)
    return measured


def count_key_to_index(bitstring):
    """Convert a Qiskit count key for c_i into the measured integer index."""
    # Qiskit muestra los bits clasicos en orden big-endian; int(..., 2) recupera c_i como entero.
    compact = str(bitstring).replace(" ", "")
    return int(compact, 2)


def build_shot_qw_case(ctx, lam=None, t_total=None, r_trotter=V7_1_DEFAULT_TROTTER_STEPS, gamma=1.0, mode="adjacency"):
    """Build one hardware-layout circuit for finite-shot execution."""
    # Esta ruta no calcula statevector; solo construye el circuito medible.
    solution_for_report = reporting_solution_indices(ctx)
    if not solution_for_report:
        raise ValueError("No valid windows in this case.")
    if lam is None:
        lam = np.sqrt(ctx["W"]) * float(gamma)
    if t_total is None:
        t_total = np.pi / 4.0 * np.sqrt(ctx["W"] / len(solution_for_report))
    qc, metadata = build_hardware_qw_circuit(
        ctx, lam=lam, t_total=t_total, r_trotter=r_trotter, gamma=gamma, mode=mode
    )
    if qc.num_qubits > V7_1_MAX_SIM_QUBITS:
        raise ValueError(f"Circuit has {qc.num_qubits} qubits; shot-simulation cap is {V7_1_MAX_SIM_QUBITS}.")
    return {
        "case": ctx["name"],
        "ctx": ctx,
        "qc": qc,
        "metadata": metadata,
        "solution_indices_reporting_only": solution_for_report,
        "P_uniform": len(solution_for_report) / ctx["W"],
        "P_valid": np.nan,
        "op_count": int(sum(qc.count_ops().values())),
    }


def run_idx_shots(result, shots=V7_1_SHOTS, seed=V7_1_SHOT_SEED):
    """Measure idx with finite shots and estimate P_valid from observed counts."""
    # El conjunto solucion se usa solo para puntuar las muestras ya obtenidas.
    measured_qc = add_index_measurements(result["qc"])
    W = result["ctx"]["W"]
    IDX = result["ctx"]["IDX"]

    if AerSimulator is not None:
        simulator = AerSimulator(method="statevector", seed_simulator=int(seed))
        counts = simulator.run(measured_qc, shots=int(shots)).result().get_counts()
        backend_label = "AerSimulator(statevector)"
    else:
        if "probs" not in result:
            raise ImportError("qiskit-aer is required for shot simulation above the statevector cap.")
        # Fallback reproducible cuando qiskit-aer no esta instalado en el kernel.
        rng = np.random.default_rng(int(seed))
        physical_probs = np.asarray(result["probs"], dtype=float)
        physical_probs = physical_probs / np.sum(physical_probs)
        sampled_counts = rng.multinomial(int(shots), physical_probs)
        counts = {format(i, f"0{IDX}b"): int(c) for i, c in enumerate(sampled_counts) if c}
        backend_label = "statevector multinomial sampler (Aer unavailable)"

    solution = set(result["solution_indices_reporting_only"])
    valid_count = 0
    physical_count = 0
    index_counts = {i: 0 for i in range(W)}
    padded_count = 0

    for key, count in counts.items():
        idx_value = count_key_to_index(key)
        if idx_value < W:
            physical_count += int(count)
            index_counts[idx_value] += int(count)
            if idx_value in solution:
                valid_count += int(count)
        else:
            padded_count += int(count)

    p_valid_shots = valid_count / float(shots)
    stderr = np.sqrt(max(p_valid_shots * (1.0 - p_valid_shots), 0.0) / float(shots))
    return {
        "case": result["case"],
        "mode": result["metadata"]["mode"],
        "qubits": int(result["qc"].num_qubits),
        "op_count": int(result.get("op_count", sum(result["qc"].count_ops().values()))),
        "shots": int(shots),
        "backend": backend_label,
        "measured_qc": measured_qc,
        "counts": counts,
        "index_counts": index_counts,
        "valid_count": int(valid_count),
        "physical_count": int(physical_count),
        "padded_count": int(padded_count),
        "P_valid_shots": float(p_valid_shots),
        "P_valid_statevector": float(result.get("P_valid", np.nan)),
        "P_uniform": float(result["P_uniform"]),
        "stderr": float(stderr),
    }


SHOT_RESULTS = []
SHOT_SKIPPED = []
STATEVECTOR_LOOKUP = {
    (res["case"], res["metadata"]["mode"]): res["P_valid"]
    for res in SIM_RESULTS
}

for ctx in CASE_CONTEXTS:
    total_qubits = ctx["GRID"] + ctx["IDX"] + ctx["WIN"]
    if total_qubits > V7_1_MAX_SIM_QUBITS:
        reason = f"qubits {total_qubits} > cap {V7_1_MAX_SIM_QUBITS}"
        print(f"skip shot simulation: {ctx['name']} | {reason}")
        SHOT_SKIPPED.append({"case": ctx["name"], "mode": "both", "qubits": total_qubits, "op_count": np.nan, "reason": reason})
        continue
    for mode in ["adjacency", "laplacian"]:
        shot_input = build_shot_qw_case(ctx, mode=mode)
        op_count = int(shot_input["op_count"])
        if shot_input["qc"].num_qubits > V7_1_MAX_STATEVECTOR_QUBITS:
            reason = f"exact statevector shots limited to {V7_1_MAX_STATEVECTOR_QUBITS} qubits; case has {shot_input['qc'].num_qubits}"
            print(f"skip shot simulation: {ctx['name']} | mode={mode} | qubits={shot_input['qc'].num_qubits} | {reason}")
            SHOT_SKIPPED.append({"case": ctx["name"], "mode": mode, "qubits": shot_input["qc"].num_qubits, "op_count": op_count, "reason": reason})
            continue
        if op_count > V7_1_MAX_AER_OPS:
            reason = f"ops {op_count} > cap {V7_1_MAX_AER_OPS}"
            print(f"skip shot simulation: {ctx['name']} | mode={mode} | qubits={shot_input['qc'].num_qubits} | {reason}")
            SHOT_SKIPPED.append({"case": ctx["name"], "mode": mode, "qubits": shot_input["qc"].num_qubits, "op_count": op_count, "reason": reason})
            continue
        shot_input["P_valid"] = STATEVECTOR_LOOKUP.get((ctx["name"], mode), np.nan)
        active_shots = V7_1_SHOTS
        try:
            shot_result = run_idx_shots(shot_input, shots=active_shots)
        except Exception as exc:
            reason = f"{type(exc).__name__}: {exc}"
            print(f"shot simulation failed: {ctx['name']} | mode={mode} | qubits={shot_input['qc'].num_qubits} | ops={op_count} | {reason}")
            SHOT_SKIPPED.append({"case": ctx["name"], "mode": mode, "qubits": shot_input["qc"].num_qubits, "op_count": op_count, "reason": reason})
            continue
        SHOT_RESULTS.append(shot_result)
        statevector_text = "n/a" if np.isnan(shot_result["P_valid_statevector"]) else f"{shot_result['P_valid_statevector']:.4f}"
        print(
            f"{shot_result['case']} | mode={shot_result['mode']} | qubits={shot_result['qubits']} | ops={shot_result['op_count']} | shots={shot_result['shots']} | "
            f"valid counts={shot_result['valid_count']} | P_valid(shots)={shot_result['P_valid_shots']:.4f} ± {shot_result['stderr']:.4f} | "
            f"statevector={statevector_text} | padded={shot_result['padded_count']} | backend={shot_result['backend']}"
        )

csv_path = V7_OUTPUT_DIR / "v7_1_shot_results.csv"
shot_headers = [
    "case", "mode", "qubits", "op_count", "shots", "backend", "valid_count", "physical_count", "padded_count",
    "P_uniform", "P_valid_shots", "stderr", "P_valid_statevector",
]
with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=shot_headers)
    writer.writeheader()
    for row in SHOT_RESULTS:
        writer.writerow({key: row[key] for key in shot_headers})
print(f"Saved {csv_path}")

skipped_path = V7_OUTPUT_DIR / "v7_1_shot_skipped.csv"
skipped_headers = ["case", "mode", "qubits", "op_count", "reason"]
with skipped_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=skipped_headers)
    writer.writeheader()
    writer.writerows(SHOT_SKIPPED)
print(f"Saved {skipped_path}")


01_1d_tiny_single_gap | mode=adjacency | qubits=11 | ops=204 | shots=4096 | valid counts=2201 | P_valid(shots)=0.5374 ± 0.0078 | statevector=0.5325 | padded=0 | backend=AerSimulator(statevector)


01_1d_tiny_single_gap | mode=laplacian | qubits=11 | ops=304 | shots=4096 | valid counts=2284 | P_valid(shots)=0.5576 ± 0.0078 | statevector=0.5597 | padded=0 | backend=AerSimulator(statevector)


02_1d_main_reference | mode=adjacency | qubits=13 | ops=262 | shots=4096 | valid counts=2334 | P_valid(shots)=0.5698 ± 0.0077 | statevector=0.5668 | padded=0 | backend=AerSimulator(statevector)


02_1d_main_reference | mode=laplacian | qubits=13 | ops=386 | shots=4096 | valid counts=2065 | P_valid(shots)=0.5042 ± 0.0078 | statevector=0.4923 | padded=0 | backend=AerSimulator(statevector)


03_1d_two_free_regions | mode=adjacency | qubits=16 | ops=354 | shots=4096 | valid counts=2745 | P_valid(shots)=0.6702 ± 0.0073 | statevector=0.6738 | padded=0 | backend=AerSimulator(statevector)


...

06_2d_tiny_corner_block | mode=adjacency | qubits=15 | ops=236 | shots=4096 | valid counts=3199 | P_valid(shots)=0.7810 ± 0.0065 | statevector=0.7798 | padded=0 | backend=AerSimulator(statevector)


06_2d_tiny_corner_block | mode=laplacian | qubits=15 | ops=284 | shots=4096 | valid counts=3199 | P_valid(shots)=0.7810 ± 0.0065 | statevector=0.7798 | padded=0 | backend=AerSimulator(statevector)
skip shot simulation: 07_2d_small_diagonal_block | mode=adjacency | qubits=24 | exact statevector shots limited to 16 qubits; case has 24
skip shot simulation: 07_2d_small_diagonal_block | mode=laplacian | qubits=24 | exact statevector shots limited to 16 qubits; case has 24
skip shot simulation: 08_2d_medium_clustered_obstacles | mode=adjacency | qubits=33 | exact statevector shots limited to 16 qubits; case has 33


skip shot simulation: 08_2d_medium_clustered_obstacles | mode=laplacian | qubits=33 | exact statevector shots limited to 16 qubits; case has 33
skip shot simulation: 09_2d_rectangular_window | mode=adjacency | qubits=47 | exact statevector shots limited to 16 qubits; case has 47


skip shot simulation: 09_2d_rectangular_window | mode=laplacian | qubits=47 | exact statevector shots limited to 16 qubits; case has 47
skip shot simulation: 10_3d_small_clustered_obstacles | mode=adjacency | qubits=61 | exact statevector shots limited to 16 qubits; case has 61


skip shot simulation: 10_3d_small_clustered_obstacles | mode=laplacian | qubits=61 | exact statevector shots limited to 16 qubits; case has 61
Saved TFG_V7_1_Analysis/v7_1_shot_results.csv
Saved TFG_V7_1_Analysis/v7_1_shot_skipped.csv


## Shot-Based Probability Comparison

This figure uses the finite-shot frequencies from the previous cell. Error bars show one binomial standard error, so repeated runs with different random seeds will fluctuate around the statevector reference.


In [12]:
if SHOT_RESULTS:
    cases = sorted({res["case"] for res in SHOT_RESULTS})
    x = np.arange(len(cases))
    width = 0.25
    adj = {res["case"]: res for res in SHOT_RESULTS if res["mode"] == "adjacency"}
    lap = {res["case"]: res for res in SHOT_RESULTS if res["mode"] == "laplacian"}
    uniform_vals = np.array([adj.get(c, lap.get(c))["P_uniform"] for c in cases])
    adj_vals = np.array([adj[c]["P_valid_shots"] if c in adj else np.nan for c in cases])
    lap_vals = np.array([lap[c]["P_valid_shots"] if c in lap else np.nan for c in cases])
    adj_err = np.array([adj[c]["stderr"] if c in adj else np.nan for c in cases])
    lap_err = np.array([lap[c]["stderr"] if c in lap else np.nan for c in cases])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width, uniform_vals, width=width, color=BASELINE_COLOR, alpha=0.75, label="uniform")
    ax.bar(x, adj_vals, width=width, yerr=adj_err, capsize=4, color=VALID_COLOR, alpha=0.9, label="adjacency shots")
    ax.bar(x + width, lap_vals, width=width, yerr=lap_err, capsize=4, color="#3498db", alpha=0.85, label="laplacian shots")
    ax.set_xticks(x)
    ax.set_xticklabels([c.split("_", 1)[0] for c in cases])
    ax.set_xlabel("case")
    ax.set_ylabel("observed P_valid")
    ax.set_title(f"V7.1 measured idx frequencies ({V7_1_SHOTS} shots)")
    ax.legend(fontsize=10)
    fig.tight_layout()
    save_qw_figure(fig, "v7_1_shot_probability_comparison")
else:
    print("No shot results available.")


## Resource Estimate for All Cases

This cell builds circuits for every case and reports qubit counts and operation counts. It does not simulate large statevectors. This is the practical view needed before transpilation to a specific backend.


In [13]:
RESOURCE_ROWS = []
for ctx in CASE_CONTEXTS:
    for mode in ["adjacency", "laplacian"]:
        solution_for_report = reporting_solution_indices(ctx)
        lam = np.sqrt(ctx["W"])
        t_total = np.pi / 4.0 * np.sqrt(ctx["W"] / len(solution_for_report))
        qc, meta = build_hardware_qw_circuit(ctx, lam=lam, t_total=t_total, r_trotter=V7_1_DEFAULT_TROTTER_STEPS, mode=mode)
        row = {
            "case": ctx["name"],
            "mode": mode,
            "W": ctx["W"],
            "K_reporting_only": len(solution_for_report),
            "grid_qubits": ctx["GRID"],
            "idx_qubits": ctx["IDX"],
            "work_qubits": ctx["WIN"],
            "total_qubits": qc.num_qubits,
            "r_trotter": V7_1_DEFAULT_TROTTER_STEPS,
            "operations": str(meta["operations"]),
        }
        RESOURCE_ROWS.append(row)
        print(row)

csv_path = V7_OUTPUT_DIR / "v7_1_hardware_resources.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(RESOURCE_ROWS[0].keys()))
    writer.writeheader()
    writer.writerows(RESOURCE_ROWS)
print(f"Saved {csv_path}")


{'case': '01_1d_tiny_single_gap', 'mode': 'adjacency', 'W': 5, 'K_reporting_only': 1, 'grid_qubits': 6, 'idx_qubits': 3, 'work_qubits': 2, 'total_qubits': 11, 'r_trotter': 4, 'operations': "{'x': 99, 'mcx': 80, 'unitary': 16, 'cp': 4, 'barrier': 4, 'initialize': 1}"}
{'case': '01_1d_tiny_single_gap', 'mode': 'laplacian', 'W': 5, 'K_reporting_only': 1, 'grid_qubits': 6, 'idx_qubits': 3, 'work_qubits': 2, 'total_qubits': 11, 'r_trotter': 4, 'operations': "{'x': 179, 'mcx': 80, 'mcphase': 20, 'unitary': 16, 'cp': 4, 'barrier': 4, 'initialize': 1}"}
{'case': '02_1d_main_reference', 'mode': 'adjacency', 'W': 7, 'K_reporting_only': 2, 'grid_qubits': 8, 'idx_qubits': 3, 'work_qubits': 2, 'total_qubits': 13, 'r_trotter': 4, 'operations': "{'x': 117, 'mcx': 112, 'unitary': 24, 'cp': 4, 'barrier': 4, 'initialize': 1}"}
{'case': '02_1d_main_reference', 'mode': 'laplacian', 'W': 7, 'K_reporting_only': 2, 'grid_qubits': 8, 'idx_qubits': 3, 'work_qubits': 2, 'total_qubits': 13, 'r_trotter': 4, 'oper

{'case': '04_1d_clustered_medium', 'mode': 'laplacian', 'W': 14, 'K_reporting_only': 4, 'grid_qubits': 16, 'idx_qubits': 4, 'work_qubits': 3, 'total_qubits': 23, 'r_trotter': 4, 'operations': "{'x': 455, 'mcx': 336, 'mcphase': 60, 'unitary': 52, 'barrier': 4, 'initialize': 1}"}
{'case': '05_1d_long_clustered_blocks', 'mode': 'adjacency', 'W': 29, 'K_reporting_only': 11, 'grid_qubits': 32, 'idx_qubits': 5, 'work_qubits': 4, 'total_qubits': 41, 'r_trotter': 4, 'operations': "{'mcx': 928, 'x': 348, 'unitary': 112, 'mcphase': 4, 'barrier': 4, 'initialize': 1}"}


{'case': '05_1d_long_clustered_blocks', 'mode': 'laplacian', 'W': 29, 'K_reporting_only': 11, 'grid_qubits': 32, 'idx_qubits': 5, 'work_qubits': 4, 'total_qubits': 41, 'r_trotter': 4, 'operations': "{'x': 972, 'mcx': 928, 'mcphase': 120, 'unitary': 112, 'barrier': 4, 'initialize': 1}"}
{'case': '06_2d_tiny_corner_block', 'mode': 'adjacency', 'W': 4, 'K_reporting_only': 1, 'grid_qubits': 9, 'idx_qubits': 2, 'work_qubits': 4, 'total_qubits': 15, 'r_trotter': 4, 'operations': "{'mcx': 128, 'x': 83, 'unitary': 16, 'mcphase': 4, 'barrier': 4, 'initialize': 1}"}
{'case': '06_2d_tiny_corner_block', 'mode': 'laplacian', 'W': 4, 'K_reporting_only': 1, 'grid_qubits': 9, 'idx_qubits': 2, 'work_qubits': 4, 'total_qubits': 15, 'r_trotter': 4, 'operations': "{'mcx': 128, 'x': 115, 'cp': 16, 'unitary': 16, 'mcphase': 4, 'barrier': 4, 'initialize': 1}"}
{'case': '07_2d_small_diagonal_block', 'mode': 'adjacency', 'W': 9, 'K_reporting_only': 1, 'grid_qubits': 16, 'idx_qubits': 4, 'work_qubits': 4, 'tota

{'case': '08_2d_medium_clustered_obstacles', 'mode': 'laplacian', 'W': 16, 'K_reporting_only': 9, 'grid_qubits': 25, 'idx_qubits': 4, 'work_qubits': 4, 'total_qubits': 33, 'r_trotter': 4, 'operations': "{'mcx': 512, 'x': 470, 'unitary': 96, 'mcphase': 68, 'barrier': 4, 'initialize': 1}"}
{'case': '09_2d_rectangular_window', 'mode': 'adjacency', 'W': 20, 'K_reporting_only': 8, 'grid_qubits': 36, 'idx_qubits': 5, 'work_qubits': 6, 'total_qubits': 47, 'r_trotter': 4, 'operations': "{'mcx': 960, 'x': 295, 'unitary': 124, 'mcphase': 4, 'barrier': 4, 'initialize': 1}"}


{'case': '09_2d_rectangular_window', 'mode': 'laplacian', 'W': 20, 'K_reporting_only': 8, 'grid_qubits': 36, 'idx_qubits': 5, 'work_qubits': 6, 'total_qubits': 47, 'r_trotter': 4, 'operations': "{'mcx': 960, 'x': 775, 'unitary': 124, 'mcphase': 84, 'barrier': 4, 'initialize': 1}"}
{'case': '10_3d_small_clustered_obstacles', 'mode': 'adjacency', 'W': 18, 'K_reporting_only': 12, 'grid_qubits': 48, 'idx_qubits': 5, 'work_qubits': 8, 'total_qubits': 61, 'r_trotter': 4, 'operations': "{'mcx': 1152, 'x': 294, 'unitary': 132, 'mcphase': 4, 'barrier': 4, 'initialize': 1}"}


{'case': '10_3d_small_clustered_obstacles', 'mode': 'laplacian', 'W': 18, 'K_reporting_only': 12, 'grid_qubits': 48, 'idx_qubits': 5, 'work_qubits': 8, 'total_qubits': 61, 'r_trotter': 4, 'operations': "{'mcx': 1152, 'x': 734, 'unitary': 132, 'mcphase': 76, 'barrier': 4, 'initialize': 1}"}
Saved TFG_V7_1_Analysis/v7_1_hardware_resources.csv
